# verse and poem clustering pipeline

this notebook runs the complete workflow for clustering verses and poems using embeddings and graph-based methods.

## setup and configuration

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')

from config import Config
from pipeline import Pipeline

## configure parameters

In [2]:
config = Config()

config.csv_path = '../concatenated.csv'
config.output_dir = '/scratch/gent/vo/000/gvo00042/vsc48660'
config.use_scratch = False

config.subset_fraction = 0.01
config.batch_size = 32

config.verse_similarity_thresholds = [0.7, 0.75, 0.8, 0.85, 0.9]
config.leiden_resolutions = [0.5, 1.0, 1.5, 2.0]
config.n_bootstraps = 5
config.n_gridsearch_iterations = 5

config.poem_similarity_thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

print(f'configured to use {config.n_cores} cpu cores')
print(f'output directory: {config.ensure_output_dir()}')

configured to use 30 cpu cores
output directory: .


## run the complete pipeline

In [3]:
pipeline = Pipeline(config)
results = pipeline.run()

starting verse and poem clustering pipeline

step 1: loading data
loaded 2351003 verses from 263202 poems

step 2: creating stratified subset
created subset with 23082 verses

step 3: loading embedding model


2025-11-27 09:32:38.128776: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


loaded model on cuda

step 4: embedding verses


embedding: 100%|██████████| 722/722 [00:57<00:00, 12.66it/s]


generated embeddings with dimension 768

step 5: building faiss index
built IndexIVFFlat index

step 6: running verse gridsearch

verse gridsearch results:
   threshold  resolution  n_pairs  n_clusters  modularity  stability  \
0       0.70         0.5  1156616           8    0.620964   0.902382   
1       0.70         1.0  1156616          17    0.642872   0.880540   
2       0.70         1.5  1156616          23    0.622655   0.694784   
3       0.70         2.0  1156616          29    0.607347   0.754650   
4       0.75         0.5  1155814           7    0.613797   0.799389   

   iteration_time  
0       88.551904  
1       90.508935  
2      116.812211  
3      128.337822  
4       85.167641  

best parameters:
threshold         7.000000e-01
resolution        5.000000e-01
n_pairs           1.156616e+06
n_clusters        8.000000e+00
modularity        6.209640e-01
stability         9.023822e-01
iteration_time    8.855190e+01
Name: 0, dtype: float64
saved verse gridsearch visualiza

embedding:  60%|█████▉    | 44013/73469 [41:16<27:37, 17.77it/s]  


KeyboardInterrupt: 

## inspect results

In [ ]:
verse_clusters = results['verse_clusters']
poem_clusters = results['poem_clusters']

print(f'number of verse clusters: {len(set(verse_clusters.values()))}')
print(f'number of poem clusters: {len(set(poem_clusters.values()))}')
print(f'number of verses clustered: {len(verse_clusters)}')
print(f'number of poems clustered: {len(poem_clusters)}')

## optional: run individual stages

### stage 1: verse clustering on subset

In [ ]:
best_verse_params = pipeline.run_verse_clustering_on_subset()
print(best_verse_params)

### stage 2: verse clustering on full dataset

In [ ]:
if best_verse_params is not None:
    verse_clusters = pipeline.run_full_verse_clustering(
        best_verse_params['threshold'],
        best_verse_params['resolution']
    )
else:
    verse_clusters = pipeline.run_full_verse_clustering(0.8, 1.0)

### stage 3: poem clustering on subset

In [ ]:
best_poem_params = pipeline.run_poem_clustering_on_subset()
print(best_poem_params)

### stage 4: poem clustering on full dataset

In [ ]:
if best_poem_params is not None:
    poem_clusters = pipeline.run_full_poem_clustering(best_poem_params['threshold'])
else:
    poem_clusters = pipeline.run_full_poem_clustering(0.5)

## save results

In [ ]:
import pandas as pd
import os

output_dir = config.ensure_output_dir()

verse_df = pd.DataFrame([
    {'verse_id': k, 'cluster': v} 
    for k, v in verse_clusters.items()
])
verse_df.to_csv(os.path.join(output_dir, 'verse_clusters.csv'), index=False)

poem_df = pd.DataFrame([
    {'poem_id': k, 'cluster': v} 
    for k, v in poem_clusters.items()
])
poem_df.to_csv(os.path.join(output_dir, 'poem_clusters.csv'), index=False)

print(f'saved verse clusters to {output_dir}/verse_clusters.csv')
print(f'saved poem clusters to {output_dir}/poem_clusters.csv')